# Evaluate ai.extract(...) Quality with PySpark

This notebook evaluates named-entity extraction for factual support and coverage. The AI transformations and LLM-as-a-Judge evaluation remain in Spark. Only the small, materialized result set is converted to pandas for metrics and charts.

### What You'll Do
1. Extract people, organizations, and locations from sample text.
2. Use a fixed judge to score consistency and relevance.
3. Inspect average and per-sample quality.
4. Compare plain string labels with richer `ExtractLabel` definitions.

### Before You Start
- **Runtime** - This notebook was made for **Fabric 1.3 runtime**.
- **Customize it** - Replace the sample data and adapt the judge criteria to your use case.
- **Keep comparisons fair** - Hold the judge model and prompts fixed while changing one executor setting at a time.
- **Validate important decisions** - LLM judge scores are useful proxies, not a substitute for human-reviewed production samples.

| Metric | Measures |
|--------|----------|
| **Consistency** | Extracted entities are explicitly supported by the source |
| **Relevance** | Important source entities are captured |

[ai.extract PySpark documentation](https://learn.microsoft.com/fabric/data-science/ai-functions/pyspark/extract)


## 1. Setup

Install Pydantic for the structured response schemas used by the judge.
The executor uses `gpt-5-mini` with low reasoning effort. Each metric is
scored independently by a fixed `gpt-5.1` judge.


In [ ]:
%pip install -q pydantic 2>/dev/null


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import synapse.ml.spark.aifunc as aifunc
from pydantic import BaseModel, Field
from pyspark.sql import functions as F

# Use the smaller model for the function under test and a larger fixed model for judging.
EXECUTOR_OPTIONS = {
    "deploymentName": "gpt-5-mini",
    "reasoningEffort": "low",
}
JUDGE_OPTIONS = {
    "deploymentName": "gpt-5.1",
    "reasoningEffort": "medium",
}

class MetricEval(BaseModel):
    reason: str = Field(description="Brief rationale for the score")
    score: int = Field(ge=1, le=5, description="Integer score from 1 to 5")

def materialize(frame):
    cached = frame.cache()
    _ = cached.count()
    error_columns = [name for name in cached.columns if name.endswith("_error")]
    if error_columns:
        has_error = F.lit(False)
        for name in error_columns:
            has_error = has_error | F.coalesce(
                F.length(F.trim(F.col(name).cast("string"))) > 0,
                F.lit(False),
            )
        failed_rows = cached.filter(has_error)
        failure_count = failed_rows.count()
        if failure_count:
            print(f"{failure_count} row(s) contain AI Function errors:")
            id_columns = [
                name for name in ("sample_id", "ticket_id") if name in cached.columns
            ]
            display(failed_rows.select(*id_columns, *error_columns))
    return cached

def fresh_ai_view(frame):
    return frame.select("*")

def add_judge_metric(frame, metric_name, prompt, column_prefix=""):
    score_col = f"{column_prefix}{metric_name}"
    response_col = f"_{score_col}_response"
    raw_score_col = f"_{score_col}_raw_score"
    error_col = f"_{score_col}_error"
    judged = fresh_ai_view(frame).ai.generate_response(
        prompt=prompt,
        is_prompt_template=True,
        output_col=response_col,
        error_col=error_col,
        response_format=MetricEval,
        **JUDGE_OPTIONS,
    )
    invalid_score = (
        F.col(raw_score_col).isNull()
        | ~F.col(raw_score_col).between(1, 5)
        | (F.col(raw_score_col) != F.floor(F.col(raw_score_col)))
    )
    validation_message = "Judge score must be an integer from 1 to 5"
    return (
        judged
        .withColumn(
            raw_score_col,
            F.get_json_object(F.col(response_col), "$.score").cast("double"),
        )
        .withColumn(
            error_col,
            F.when(
                invalid_score,
                F.when(
                    F.length(
                        F.trim(F.coalesce(F.col(error_col), F.lit("")))
                    ) > 0,
                    F.concat(
                        F.col(error_col),
                        F.lit(f"; {validation_message}"),
                    ),
                ).otherwise(F.lit(validation_message)),
            ).otherwise(F.col(error_col)),
        )
        .withColumn(
            score_col,
            F.when(
                ~invalid_score,
                F.col(raw_score_col).cast("int"),
            ),
        )
        .withColumn(
            f"{score_col}_reason",
            F.get_json_object(F.col(response_col), "$.reason"),
        )
        .drop(raw_score_col)
    )


## 2. Load Sample Data


In [ ]:
ENTITY_LABELS = ['person', 'organization', 'location']
rows = [(1,
  'In Seattle, Microsoft vice president Sarah Chen announced that Contoso Retail signed a cloud support '
  'agreement with Microsoft.'),
 (2,
  'At a press briefing in Geneva, WHO director Dr. Amina Yusuf said UNICEF will run a measles campaign with '
  'the Kenya Ministry of Health.'),
 (3,
  "In Toronto, Justice Elena Park of the Ontario Superior Court approved CleanGrid Energy's settlement with "
  'Northwind Power after a six-month review.'),
 (4,
  'In Singapore, DBS Bank CEO Piyush Gupta confirmed that DBS Bank signed a fraud analytics partnership with '
  'OpenAI.'),
 (5,
  'In Berlin, Dr. Lena Vogel from the Max Planck Institute and engineer Jonas Weber from Siemens '
  'Healthineers presented a new MRI calibration study.')]
df = spark.createDataFrame(rows, ["sample_id", "text"])
display(df)


## 3. Run `ai.extract`


In [ ]:
baseline_df = materialize(
    df.ai.extract(
        labels=ENTITY_LABELS,
        input_col="text",
        error_col="executor_error",
        **EXECUTOR_OPTIONS,
    )
)
display(baseline_df.select("text", *ENTITY_LABELS))
display(baseline_df.ai.stats)


## 4. Evaluate with an LLM Judge


In [ ]:
EVAL_METRICS = {
    "consistency": """Score factual consistency from 1 to 5.
A score of 5 means every extracted entity is explicitly present in the source.
Penalize fabricated, unsupported, or incorrectly typed entities.

<source_text>
{text}
</source_text>
<extracted_entities>
{_extracted_summary}
</extracted_entities>""",
    "relevance": """Score entity coverage from 1 to 5.
A score of 5 means the extraction captures all important named people,
organizations, and locations needed to understand the source.
Penalize missing key entities.

<source_text>
{text}
</source_text>
<extracted_entities>
{_extracted_summary}
</extracted_entities>""",
}

evaluated_df = fresh_ai_view(baseline_df).withColumn(
    "_extracted_summary",
    F.to_json(F.struct(*(F.col(name).alias(name) for name in ENTITY_LABELS))),
)
for metric_name, prompt in EVAL_METRICS.items():
    evaluated_df = add_judge_metric(evaluated_df, metric_name, prompt)

evaluated_df = materialize(evaluated_df)
display(evaluated_df.select("text", *ENTITY_LABELS, *EVAL_METRICS.keys()))


## 5. Results


In [ ]:
METRICS = ['consistency', 'relevance']
results_pd = evaluated_df.select('sample_id', 'text', 'person', 'organization', 'location', 'consistency', 'relevance').toPandas()

score_summary = pd.DataFrame({
    "Metric": ['Consistency', 'Relevance'],
    "Average score": [results_pd[metric].mean() for metric in METRICS],
    "Scored rows": [results_pd[metric].notna().sum() for metric in METRICS],
})
def quality_status(score, is_complete):
    if not is_complete or pd.isna(score):
        return "INCOMPLETE"
    return "PASS" if score >= 4 else "REVIEW" if score >= 3.5 else "FAIL"

score_summary["Status"] = [
    quality_status(score, scored_rows == len(results_pd))
    for score, scored_rows in zip(
        score_summary["Average score"],
        score_summary["Scored rows"],
    )
]
display(score_summary.round(2))

labels = score_summary["Metric"].tolist()
values = score_summary["Average score"].tolist()
fig = plt.figure(figsize=(13, 4.5))
bar_ax = fig.add_subplot(1, 2, 1)
bars = bar_ax.bar(labels, values, color="#0077aa")
bar_ax.set_ylim(0, 5)
bar_ax.set_ylabel("Score (1-5)")
bar_ax.set_title('Entity Extraction Quality')
bar_ax.axhline(y=4, color="#999999", linestyle="--", alpha=0.6)
bar_ax.tick_params(axis="x", rotation=20)
bar_ax.bar_label(bars, fmt="%.2f", padding=2)

if len(METRICS) >= 3:
    detail_ax = fig.add_subplot(1, 2, 2, polar=True)
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    detail_ax.plot(
        angles + angles[:1],
        values + values[:1],
        "o-",
        linewidth=2,
        color="#9955bb",
    )
    detail_ax.fill(
        angles + angles[:1],
        values + values[:1],
        alpha=0.25,
        color="#9955bb",
    )
    detail_ax.set_xticks(angles)
    detail_ax.set_xticklabels(labels)
    detail_ax.set_ylim(0, 5)
    detail_ax.set_title("Quality Profile", pad=20)
else:
    detail_ax = fig.add_subplot(1, 2, 2)
    detail_ax.hist(
        [results_pd[metric].dropna() for metric in METRICS],
        bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5],
        label=labels,
        alpha=0.7,
    )
    detail_ax.set_xticks([1, 2, 3, 4, 5])
    detail_ax.set_xlabel("Score")
    detail_ax.set_ylabel("Rows")
    detail_ax.set_title("Score Distribution")
    detail_ax.legend()
plt.tight_layout()
plt.show()

results_pd["scored_metrics"] = results_pd[METRICS].notna().sum(axis=1)
complete_rows = results_pd["scored_metrics"].eq(len(METRICS))
results_pd["average_score"] = (
    results_pd[METRICS].mean(axis=1).where(complete_rows).round(2)
)
results_pd["status"] = [
    quality_status(score, is_complete)
    for score, is_complete in zip(
        results_pd["average_score"],
        complete_rows,
    )
]


In [ ]:
breakdown_pd = results_pd[
    ["text", "consistency", "relevance", "scored_metrics", "average_score", "status"]
].copy()
breakdown_pd["text"] = breakdown_pd["text"].str[:120] + "..."
display(breakdown_pd)


## 6. Optional Refinement: Richer `ExtractLabel` Definitions

Add descriptions, types, cardinality limits, and a role field, then evaluate
the refined extraction with the same judge prompts.


In [ ]:
advanced_fields = ENTITY_LABELS + ["role"]
advanced_df = materialize(
    df.ai.extract(
        labels=[
            aifunc.ExtractLabel(
                label="person",
                description="Distinct people explicitly named, including titles when present",
                type="string",
                max_items=6,
            ),
            aifunc.ExtractLabel(
                label="organization",
                description="Explicitly named companies, agencies, and institutions",
                type="string",
                max_items=6,
            ),
            aifunc.ExtractLabel(
                label="location",
                description="Explicitly named cities, countries, and geographic locations",
                type="string",
                max_items=4,
            ),
            aifunc.ExtractLabel(
                label="role",
                description="Professional or institutional roles tied to named people",
                type="string",
                max_items=6,
            ),
        ],
        input_col="text",
        error_col="advanced_executor_error",
        **EXECUTOR_OPTIONS,
    )
)

advanced_eval_df = fresh_ai_view(advanced_df).withColumn(
    "_extracted_summary",
    F.to_json(F.struct(*(F.col(name).alias(name) for name in advanced_fields))),
)
for metric_name, prompt in EVAL_METRICS.items():
    advanced_eval_df = add_judge_metric(
        advanced_eval_df,
        metric_name,
        prompt,
        column_prefix="advanced_",
    )
advanced_eval_df = materialize(advanced_eval_df)
display(advanced_eval_df.select("text", *advanced_fields))


In [ ]:
advanced_pd = advanced_eval_df.select(
    "sample_id", "advanced_consistency", "advanced_relevance"
).toPandas()
comparison_rows_pd = (
    results_pd[["sample_id", "consistency", "relevance"]]
    .merge(
        advanced_pd,
        on="sample_id",
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
)
required_columns = [
    "consistency",
    "relevance",
    "advanced_consistency",
    "advanced_relevance",
]
paired_mask = (
    comparison_rows_pd["_merge"].eq("both")
    & comparison_rows_pd[required_columns].notna().all(axis=1)
)
paired_count = int(paired_mask.sum())
excluded_count = int((~paired_mask).sum())
print(f"Paired rows: {paired_count} | Excluded rows: {excluded_count}")
if excluded_count:
    display(
        comparison_rows_pd.loc[
            ~paired_mask,
            ["sample_id", "_merge", *required_columns],
        ]
    )
if not paired_count:
    raise ValueError("No rows have complete baseline and ExtractLabel scores.")
paired_pd = comparison_rows_pd.loc[paired_mask]

comparison_pd = pd.DataFrame({
    "Metric": ["Consistency", "Relevance", "Overall"],
    "Baseline": [
        paired_pd["consistency"].mean(),
        paired_pd["relevance"].mean(),
        paired_pd[["consistency", "relevance"]].mean(axis=1).mean(),
    ],
    "ExtractLabel": [
        paired_pd["advanced_consistency"].mean(),
        paired_pd["advanced_relevance"].mean(),
        paired_pd[
            ["advanced_consistency", "advanced_relevance"]
        ].mean(axis=1).mean(),
    ],
})
comparison_pd["Delta"] = comparison_pd["ExtractLabel"] - comparison_pd["Baseline"]
display(comparison_pd.round(2))

plot_pd = comparison_pd[comparison_pd["Metric"] != "Overall"].set_index("Metric")
ax = plot_pd[["Baseline", "ExtractLabel"]].plot.bar(
    figsize=(8, 4),
    color=["#0077aa", "#22cc77"],
    rot=0,
)
ax.set_ylim(0, 5)
ax.set_ylabel("Average score (1-5)")
ax.set_title("Baseline vs ExtractLabel")
ax.axhline(y=4, color="#999999", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()


## Interpreting Results

| Average score | Suggested action |
|---------------|------------------|
| **4.5-5.0** | Strong candidate for production validation |
| **4.0-4.4** | Good; inspect the lowest-scoring samples |
| **3.5-3.9** | Acceptable for iteration; refine data, prompts, or labels |
| **Below 3.5** | Investigate before broader use |
| **INCOMPLETE** | One or more judge scores are missing; inspect AI Function errors |

| Metric or issue | Likely cause | Next step |
|-----------------|--------------|-----------|
| Consistency | Unsupported or hallucinated entities | Tighten label descriptions and review failures |
| Relevance | Important entities are missing | Use more specific labels or richer `ExtractLabel` definitions |
| Both metrics | Source text or labels are ambiguous | Clarify the extraction contract |

### Improving Quality

- Prefer `ExtractLabel` when labels need descriptions, types, or cardinality limits.
- Split overloaded labels into narrower fields.
- Review consistency failures first because unsupported entities can be high risk.

Keep the judge configuration fixed for comparisons, and confirm release decisions with representative human-reviewed samples.

## Learn More

- [ai.extract PySpark documentation](https://learn.microsoft.com/fabric/data-science/ai-functions/pyspark/extract)
- [AI Functions overview](https://learn.microsoft.com/fabric/data-science/ai-functions/overview)
